In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

import joblib

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
resume_df = pd.read_csv(
    "/content/drive/MyDrive/MyDrive/SmartHire/data/processed/resume_preprocessed.csv"
)

In [6]:
resume_df.head()

,Resume_str,Category,Clean_Resume
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR,hr administratormarketing associate hr adminis...
1,"HR SPECIALIST, US HR OPERATIONS ...",HR,hr specialist u hr operation summary versatile...
2,HR DIRECTOR Summary Over 2...,HR,hr director summary year experience recruiting...
3,HR SPECIALIST Summary Dedica...,HR,hr specialist summary dedicated driven dynamic...
4,HR MANAGER Skill Highlights ...,HR,hr manager skill highlight hr skill hr departm...


In [7]:
print(resume_df.shape)

(2481, 3)


In [8]:
resume_df["Category"].value_counts()

,count
Category,
INFORMATION-TECHNOLOGY,120
BUSINESS-DEVELOPMENT,119
ADVOCATE,118
CHEF,118
ENGINEERING,118
ACCOUNTANT,118
FINANCE,117
FITNESS,117
SALES,116


In [9]:
X = resume_df["Clean_Resume"]

y = resume_df["Category"]

In [10]:
print(X.head())

print(y.head())

0    hr administratormarketing associate hr adminis...
1    hr specialist u hr operation summary versatile...
2    hr director summary year experience recruiting...
3    hr specialist summary dedicated driven dynamic...
4    hr manager skill highlight hr skill hr departm...
Name: Clean_Resume, dtype: object
0    HR
1    HR
2    HR
3    HR
4    HR
Name: Category, dtype: object


In [11]:
label_encoder = LabelEncoder()

y = label_encoder.fit_transform(y)

In [12]:
mapping = pd.DataFrame({
    "Category": label_encoder.classes_,
    "Encoded_Label": range(len(label_encoder.classes_))
})

mapping

,Category,Encoded_Label
0,ACCOUNTANT,0
1,ADVOCATE,1
2,AGRICULTURE,2
3,APPAREL,3
4,ARTS,4
5,AUTOMOBILE,5
6,AVIATION,6
7,BANKING,7
8,BPO,8
9,BUSINESS-DEVELOPMENT,9


In [13]:
mapping.to_csv(
    "/content/drive/MyDrive/MyDrive/SmartHire/data/processed/category_mapping.csv",
    index=False
)

In [14]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95
)

In [15]:
X = tfidf.fit_transform(X)

In [16]:
print(X.shape)

(2481, 10000)


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [18]:
print(X_train.shape)
print(X_test.shape)

(1984, 10000)
(497, 10000)


In [19]:
!pip install xgboost

In [21]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=8,
    objective="multi:softmax",
    eval_metric="mlogloss",
    random_state=42
)

xgb.fit(X_train, y_train)

pred = xgb.predict(X_test)

print(accuracy_score(y_test, pred))

0.7947686116700201


In [22]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.75      1.00      0.86        24
           1       0.81      0.88      0.84        24
           2       0.78      0.54      0.64        13
           3       0.73      0.42      0.53        19
           4       0.57      0.57      0.57        21
           5       1.00      0.29      0.44         7
           6       0.90      0.83      0.86        23
           7       0.84      0.70      0.76        23
           8       1.00      0.75      0.86         4
           9       0.85      0.96      0.90        24
          10       0.91      0.83      0.87        24
          11       0.95      0.95      0.95        22
          12       0.79      0.65      0.71        23
          13       0.91      0.95      0.93        21
          14       0.67      0.53      0.59        19
          15       0.81      0.92      0.86        24
          16       0.80      0.67      0.73        24
          17       0.72    

In [23]:
import joblib

joblib.dump(
    xgb,
    "/content/drive/MyDrive/MyDrive/SmartHire/models/resume_classifier.pkl"
)

['/content/drive/MyDrive/MyDrive/SmartHire/models/resume_classifier.pkl']

In [24]:
joblib.dump(
    tfidf,
    "/content/drive/MyDrive/MyDrive/SmartHire/models/tfidf_vectorizer.pkl"
)

['/content/drive/MyDrive/MyDrive/SmartHire/models/tfidf_vectorizer.pkl']

In [25]:
joblib.dump(
    label_encoder,
    "/content/drive/MyDrive/MyDrive/SmartHire/models/label_encoder.pkl"
)

['/content/drive/MyDrive/MyDrive/SmartHire/models/label_encoder.pkl']